# PotholeIQ — YOLOv8n Object Detection Training (Colab GPU)

This notebook trains YOLOv8n on the Kaggle Pothole Dataset with real bounding box annotations on a free Colab T4 GPU.

In [ ]:
# 1. Install ultralytics and check GPU
!pip install -q ultralytics
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Clone PotholeIQ repo or download dataset
!git clone https://github.com/paras151206/PotholeIQ.git
%cd PotholeIQ

# Upload kaggle.json or set credentials to download dataset
!pip install -q kaggle
from google.colab import files
import os

if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("Upload kaggle.json:")
    uploaded = files.upload()
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json

# Download and prepare dataset
!python -c "import kaggle; kaggle.api.dataset_download_files('anggadwisunarto/potholes-detection-yolov8', path='data/yolo_raw', unzip=True)"
!python src/data/prepare_yolo_dataset.py

In [ ]:
# 3. Train YOLOv8n on GPU
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
results = model.train(
    data='data/yolo/data.yaml',
    epochs=25,
    imgsz=480,
    batch=16,
    plots=True,
    name='pothole_yolo_colab'
)

In [ ]:
# 4. Evaluate on held-out test split
best_model = YOLO('runs/detect/pothole_yolo_colab/weights/best.pt')
metrics = best_model.val(data='data/yolo/data.yaml', split='test', imgsz=480)

print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")
print(f"mAP@0.5:   {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")

In [ ]:
# 5. Download trained weights
from google.colab import files
files.download('runs/detect/pothole_yolo_colab/weights/best.pt')